In [ ]:
#!/usr/bin/env python3
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.6
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 6. Visualización de resultados — Regresión
# DESCRIPCIÓN:
#   Carga los resultados del Paso 5 y genera gráficas de error
#   para los mejores modelos. No reentrena ningún modelo.
#   Los límites de entrenamiento se leen del config.pkl generado
#   en el paso anterior — no hay TRAIN_LIMIT duplicado aquí.
#   Diagnóstico de curvas de aprendizaje coherente con Paso 5.1
#   (basado en RMSE y STD_Y, con prioridad en underfitting).
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import learning_curve, TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")

INPUT_DIR  = 'ml_results_grouped'
INPUT_NORM = 'ml_normalized_grouped'
OUTPUT_DIR = 'ml_results_grouped/plots'

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSROOM     = '1.6'
TOP_N         = 6
MAX_OCCUPANCY = 30

needs_scaling = {
    'Linear Regression', 'Ridge', 'Lasso', 'ElasticNet',
    'SVR (rbf)', 'SVR (linear)', 'SVR (poly deg2)',
    'KNN (k=3)', 'KNN (k=5)', 'KNN (k=7)', 'KNN (k=9)', 'KNN (k=11)'
}

# ===================================================================
# 1. CARGAR RESULTADOS
# ===================================================================
print("\n" + "="*70)
print("AULA 1.6 — VISUALIZACIÓN DE RESULTADOS REGRESIÓN")
print("="*70)
print("\n1. LOADING RESULTS...")

df_summary = pd.read_excel(
    os.path.join(INPUT_DIR, 'models_summary.xlsx'), index_col=0
).sort_values('RMSE', ascending=True)

df_all_preds = pd.read_excel(os.path.join(INPUT_DIR, 'all_predictions.xlsx'))
y_test = df_all_preds['Actual'].values

with open(os.path.join(INPUT_DIR, 'predictions.pkl'), 'rb') as f:
    predictions = pickle.load(f)
with open(os.path.join(INPUT_DIR, 'config.pkl'), 'rb') as f:
    config = pickle.load(f)
with open(os.path.join(INPUT_DIR, 'all_models.pkl'), 'rb') as f:
    all_models = pickle.load(f)

X_train_norm = pd.read_excel(os.path.join(INPUT_NORM, 'X_train_normalised.xlsx')).values
X_train_raw  = pd.read_excel(os.path.join(INPUT_NORM, 'X_train.xlsx')).values
y_train      = pd.read_excel(os.path.join(INPUT_NORM, 'y_train.xlsx')).values.ravel()

# ===== NUEVO: Desviación típica de y (referencia para diagnóstico) =====
STD_Y = y_train.std()
print(f"   STD_Y (referencia para diagnóstico): {STD_Y:.2f} personas")

best_model_name = config['best_model']
best_metrics    = {
    'R2':   config['best_r2'],
    'MAE':  config['best_mae'],
    'RMSE': config['best_rmse'],
}
top_models   = df_summary.index[:TOP_N].tolist()
train_limits = config.get('train_limits', {})

print(f"   Límites cargados desde config.pkl: {len(train_limits)} modelos")
print(f"   Top {TOP_N} (por RMSE ↑): {top_models}")
print(f"   Mejor modelo: {best_model_name}")

# ✅ Detectar si el paso 5 usó Val_R2 (single split) o CV_R2 (TimeSeriesSplit)
uses_val_r2 = 'Val_R2' in df_summary.columns
val_col     = 'Val_R2'  if uses_val_r2 else 'CV_R2'
val_std_col = 'Val_std' if uses_val_r2 else 'CV_std'
val_label   = 'Val R² (80/20 split)' if uses_val_r2 else 'CV R² (TimeSeriesSplit)'
print(f"   Validación interna detectada: {val_label}")

# ===================================================================
# PALETA DE COLORES
# ===================================================================
palette   = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6', '#e74c3c', '#1abc9c']
color_map = {name: palette[i] for i, name in enumerate(top_models)}

print("\n2. GENERATING PLOTS...")

# ===================================================================
# GRÁFICA 1 — Predicciones vs Reales (subplots 2x3)
# ===================================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f'Predicciones vs Valores Reales — Aula {CLASSROOM}',
            fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, name in enumerate(top_models):
    ax     = axes[i]
    y_pred = np.array(predictions[name])
    rmse   = df_summary.loc[name, 'RMSE']
    mae    = df_summary.loc[name, 'MAE']
    ax.scatter(y_test, y_pred, alpha=0.7, s=45,
            color=color_map[name], edgecolors='white', linewidth=0.5)
    lims = [min(y_test.min(), y_pred.min()) - 1,
            max(y_test.max(), y_pred.max()) + 1]
    ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.6, label='Ideal')
    ax.set_xlabel('Real (personas)', fontsize=9)
    ax.set_ylabel('Predicho (personas)', fontsize=9)
    ax.set_title(f'{name}\nRMSE={rmse:.2f} | MAE={mae:.1f}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_predictions_vs_actual_top6.png'),
            dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ 01_predictions_vs_actual_top6.png")

# ===================================================================
# GRÁFICA 2 — Distribución de residuos (violin)
# ===================================================================
fig, ax = plt.subplots(figsize=(14, 6))
residuals_data = [y_test - np.array(predictions[n]) for n in top_models]
parts = ax.violinplot(residuals_data, positions=range(len(top_models)),
                    showmedians=True, showextrema=True)
for pc, color in zip(parts['bodies'], [color_map[n] for n in top_models]):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
ax.axhline(y=0, color='red', linestyle='--', lw=2, alpha=0.8, label='Error = 0')
ax.set_xticks(range(len(top_models)))
ax.set_xticklabels(top_models, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Residuo (personas)', fontsize=11)
ax.set_title(f'Distribución de Residuos por Modelo — Aula {CLASSROOM}',
            fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_residuals_violin_top6.png'),
            dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ 02_residuals_violin_top6.png")

# ===================================================================
# GRÁFICA 3 — Comparativa RMSE, MAE, R²
# ===================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Comparativa de Métricas — Top {TOP_N} Modelos — Aula {CLASSROOM}',
            fontsize=13, fontweight='bold')
top_df  = df_summary.head(TOP_N)
metrics = [('RMSE', 'RMSE (personas)', '#e74c3c'),
        ('MAE',  'MAE (personas)',  '#3498db'),
        ('R2',   'R² Score',        '#2ecc71')]
for ax, (metric, label, color) in zip(axes, metrics):
    vals   = top_df[metric].values
    names  = top_df.index.tolist()
    colors = ['#f39c12' if i == 0 else color for i in range(len(names))]
    ax.barh(range(len(names)), vals, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel(label, fontsize=10)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    for j, v in enumerate(vals):
        ax.text(v + max(vals) * 0.01, j, f'{v:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_metrics_comparison_top6.png'),
            dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ 03_metrics_comparison_top6.png")

# ===================================================================
# GRÁFICA 4 — Error absoluto vs ocupación real (mejor modelo)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
y_pred_best = np.array(predictions[best_model_name])
abs_errors  = np.abs(y_test - y_pred_best)
scatter = ax.scatter(y_test, abs_errors, c=abs_errors, cmap='RdYlGn_r',
                    alpha=0.8, s=60, edgecolors='white', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Error absoluto (personas)')
ax.axhline(y=best_metrics['MAE'], color='navy', linestyle='--', lw=2,
        label=f"MAE = {best_metrics['MAE']:.2f}")
ax.set_xlabel('Ocupación Real (personas)', fontsize=11)
ax.set_ylabel('Error Absoluto (personas)', fontsize=11)
ax.set_title(f'Error Absoluto vs Ocupación Real\n{best_model_name} — Aula {CLASSROOM}',
            fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_abs_error_vs_occupancy.png'),
            dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ 04_abs_error_vs_occupancy.png")

# ===================================================================
# GRÁFICA 5 — Histograma de residuos con curva normal (mejor modelo)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
residuals_best = y_test - y_pred_best
ax.hist(residuals_best, bins=15, alpha=0.6, color='#3498db',
        edgecolor='white', linewidth=0.5, density=True, label='Residuos')
mu, sigma = stats.norm.fit(residuals_best)
x_range   = np.linspace(residuals_best.min(), residuals_best.max(), 200)
ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma),
        'r-', lw=2, label=f'Normal (μ={mu:.2f}, σ={sigma:.2f})')
ax.axvline(x=0, color='black', linestyle='--', lw=1.5, alpha=0.7, label='Error = 0')
ax.set_xlabel('Residuo (personas)', fontsize=11)
ax.set_ylabel('Densidad', fontsize=11)
ax.set_title(f'Distribución de Residuos — {best_model_name}\nAula {CLASSROOM}',
            fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_residuals_histogram_best.png'),
            dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ 05_residuals_histogram_best.png")

# ===================================================================
# GRÁFICA 6 — Val R² vs Test R²
# ===================================================================
fig, ax = plt.subplots(figsize=(12, 6))

def safe_val(col, name):
    if col not in df_summary.columns:
        return 0.0
    v = df_summary.loc[name, col]
    return 0.0 if pd.isna(v) else float(v)

val_r2s  = [safe_val(val_col,     n) for n in top_models]
val_stds = [safe_val(val_std_col, n) for n in top_models]
test_r2s = [df_summary.loc[n, 'R2'] for n in top_models]

x     = np.arange(len(top_models))
width = 0.35

ax.bar(x - width/2, test_r2s, width, label='R² Test',
    color='#3498db', alpha=0.85, edgecolor='white')
ax.bar(x + width/2, val_r2s,  width, label=val_label,
    color='#2ecc71', alpha=0.85, edgecolor='white',
    yerr=val_stds, capsize=5, error_kw={'linewidth': 2})

ax.set_xticks(x)
ax.set_xticklabels(top_models, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('R² Score', fontsize=11)
ax.set_title(f'R² Test vs {val_label} — Top {TOP_N} Modelos — Aula {CLASSROOM}',
            fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
all_vals = [v for v in val_r2s + test_r2s if not np.isnan(v)]
ax.set_ylim(bottom=min(0, min(all_vals) - 0.1) if all_vals else -0.1)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_val_vs_test_r2.png'),
            dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ 06_val_vs_test_r2.png")

# ===================================================================
# GRÁFICA 7 — Error Training vs Test
# ===================================================================
print("\n   Calculando errores en training...")
train_mae, train_rmse, test_mae, test_rmse = [], [], [], []

for name in top_models:
    model = all_models[name]
    X_tr  = X_train_norm if name in needs_scaling else X_train_raw
    limit = train_limits.get(name, None)
    if limit is not None:
        X_tr_fit = X_tr[:limit]
        y_tr_fit = y_train[:limit]
    else:
        X_tr_fit, y_tr_fit = X_tr, y_train

    y_pred_tr = np.clip(model.predict(X_tr_fit), 0, MAX_OCCUPANCY)
    train_mae.append(mean_absolute_error(y_tr_fit, y_pred_tr))
    train_rmse.append(np.sqrt(mean_squared_error(y_tr_fit, y_pred_tr)))
    test_mae.append(df_summary.loc[name, 'MAE'])
    test_rmse.append(df_summary.loc[name, 'RMSE'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Error en Training vs Test — Top {TOP_N} Modelos — Aula {CLASSROOM}',
            fontsize=14, fontweight='bold')
x, width = np.arange(len(top_models)), 0.35
for ax, (tr_vals, te_vals, ylabel, title) in zip(axes, [
    (train_mae,  test_mae,  'MAE (personas)',  'MAE: Training vs Test'),
    (train_rmse, test_rmse, 'RMSE (personas)', 'RMSE: Training vs Test'),
]):
    ax.bar(x - width/2, tr_vals, width, label='Train', color='#3498db', alpha=0.85, edgecolor='white')
    ax.bar(x + width/2, te_vals, width, label='Test',  color='#e74c3c', alpha=0.85, edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(top_models, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    for i, (tr, te) in enumerate(zip(tr_vals, te_vals)):
        ax.text(i - width/2, tr + 0.1, f'{tr:.1f}', ha='center', fontsize=7, color='#2c3e50')
        ax.text(i + width/2, te + 0.1, f'{te:.1f}', ha='center', fontsize=7, color='#2c3e50')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_train_vs_test_error_top6.png'),
            dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ 07_train_vs_test_error_top6.png")

# ===================================================================
# GRÁFICA 8 — Learning Curves (con diagnóstico COHERENTE con Paso 5.1)
# ===================================================================
print("\n   Calculando learning curves...")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Learning Curves — Top {TOP_N} Modelos — Aula {CLASSROOM}',
            fontsize=15, fontweight='bold')
axes = axes.flatten()
train_sizes = np.linspace(0.2, 1.0, 6)

for i, name in enumerate(top_models):
    ax    = axes[i]
    model = all_models[name]
    X_tr  = X_train_norm if name in needs_scaling else X_train_raw
    limit = train_limits.get(name, None)
    if limit is not None:
        X_tr = X_tr[:limit]
        y_tr = y_train[:limit]
    else:
        y_tr = y_train

    try:
        n_cv_lc = min(3, max(2, int(len(y_tr) * 0.2)))
        tscv    = TimeSeriesSplit(n_splits=n_cv_lc)

        train_sz, train_scores, val_scores = learning_curve(
            model, X_tr, y_tr,
            train_sizes=train_sizes,
            cv=tscv,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1 if 'Forest' in name else 1
        )
        train_mean = -train_scores.mean(axis=1)
        train_std  =  train_scores.std(axis=1)
        val_mean   = -val_scores.mean(axis=1)
        val_std    =  val_scores.std(axis=1)

        ax.plot(train_sz, train_mean, 'o-', color='#3498db', lw=2, markersize=5, label='Training RMSE')
        ax.plot(train_sz, val_mean,   'o-', color='#e67e22', lw=2, markersize=5, label='Validation RMSE')
        ax.fill_between(train_sz, train_mean - train_std, train_mean + train_std,
                        alpha=0.15, color='#3498db')
        ax.fill_between(train_sz, val_mean - val_std, val_mean + val_std,
                        alpha=0.15, color='#e67e22')
        ax.set_xlabel('Número de muestras (train)', fontsize=9)
        ax.set_ylabel('RMSE (personas)', fontsize=9)
        ax.set_title(f'Learning Curve — {name}', fontsize=10, fontweight='bold')
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)

        # ============================================================
        # DIAGNÓSTICO COHERENTE CON PASO 5.1 (basado en STD_Y)
        # ============================================================
        final_train = train_mean[-1]
        final_val   = val_mean[-1]
        gap         = final_val - final_train
        gap_ratio   = gap / final_val if final_val > 0 else 0

        # Underfitting severo
        if final_train > 1.5 * STD_Y and final_val > 1.5 * STD_Y:
            diagnosis, diag_color = '🔴 UNDERFITTING SEVERO', '#e74c3c'
        # Underfitting moderado
        elif final_train > 1.0 * STD_Y and final_val > 1.0 * STD_Y:
            diagnosis, diag_color = '🔴 UNDERFITTING', '#e67e22'
        # Overfitting severo
        elif final_train < 0.4 * STD_Y and gap_ratio > 0.5:
            diagnosis, diag_color = '🚨 OVERFITTING SEVERO', '#c0392b'
        # Overfitting
        elif gap_ratio > 0.35 or gap > 0.5 * STD_Y:
            diagnosis, diag_color = '⚠️ OVERFITTING', '#e74c3c'
        # Overfitting leve
        elif gap_ratio > 0.20 or gap > 0.25 * STD_Y:
            diagnosis, diag_color = '🟡 OVERFITTING LEVE', '#f39c12'
        # Buen ajuste
        else:
            diagnosis, diag_color = '✅ BUEN AJUSTE', '#27ae60'

        ax.text(0.97, 0.95, diagnosis, transform=ax.transAxes,
                fontsize=9, fontweight='bold', color=diag_color, ha='right', va='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=diag_color, alpha=0.85))

    except Exception as e:
        ax.text(0.5, 0.5, f'Error:\n{str(e)[:60]}',
                ha='center', va='center', transform=ax.transAxes, fontsize=8)
        ax.set_title(f'{name}', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '08_learning_curves_top6.png'),
            dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ 08_learning_curves_top6.png")

# ===================================================================
# RESUMEN FINAL
# ===================================================================
print("\n" + "="*70)
print("✅ VISUALIZACIONES COMPLETADAS")
print("="*70)
print(f"\n   📁 Gráficas en: {OUTPUT_DIR}/")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    print(f"      ✓ {fname}")
print(f"\n   🏆 Mejor modelo: {best_model_name}")
print(f"      RMSE={best_metrics['RMSE']:.2f} | MAE={best_metrics['MAE']:.2f} | R²={best_metrics['R2']:.4f}")


AULA 1.6 — VISUALIZACIÓN DE RESULTADOS REGRESIÓN

1. LOADING RESULTS...
   STD_Y (referencia para diagnóstico): 10.17 personas
   Límites cargados desde config.pkl: 2 modelos
   Top 6 (por RMSE ↑): ['KNN (k=3)', 'KNN (k=7)', 'KNN (k=5)', 'SVR (rbf)', 'SVR (linear)', 'KNN (k=9)']
   Mejor modelo: KNN (k=3)
   Validación interna detectada: Val R² (80/20 split)

2. GENERATING PLOTS...
   ✓ 01_predictions_vs_actual_top6.png
   ✓ 02_residuals_violin_top6.png
   ✓ 03_metrics_comparison_top6.png
   ✓ 04_abs_error_vs_occupancy.png
   ✓ 05_residuals_histogram_best.png
   ✓ 06_val_vs_test_r2.png

   Calculando errores en training...
   ✓ 07_train_vs_test_error_top6.png

   Calculando learning curves...
   ✓ 08_learning_curves_top6.png

✅ VISUALIZACIONES COMPLETADAS

   📁 Gráficas en: ml_results_grouped/plots/
      ✓ 01_predictions_vs_actual_top6.png
      ✓ 02_residuals_violin_top6.png
      ✓ 03_metrics_comparison_top6.png
      ✓ 04_abs_error_vs_occupancy.png
      ✓ 05_residuals_histogram_b